# TREV: Autograd vs Parameter-Shift Gradient Benchmark

Compares two gradient methods for tensor ring VQE:
1. **Parameter-shift**: 2P circuit evaluations (exact, standard approach)
2. **Autograd (backprop)**: 1 forward + 1 backward pass (exact, new approach)

Both give identical gradients. Autograd is faster for large circuits.

In [ ]:
# Install TREV (run once)
!pip install -q git+https://github.com/keunjunpark/TREV.git@optimize_kronecker

In [ ]:
import torch
import time
import numpy as np

from TREV.circuit import Circuit
from TREV.hamiltonian.hamiltonian import Hamiltonian
from TREV.measure.enums import MeasureMethod
from TREV.measure.efficient_contraction import expectation_value_batch as ev_exact
from TREV.optimization.gradients.autograd_gradient import AutogradGradient, autograd_gradient
from TREV.optimization.gradients.batch_parameter_shift import (
    BatchParameterShiftGradient, batch_gradient,
)
from TREV.optimization.optimizer import Optimizer
from TREV.optimization.optimization import minimize

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
if device == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'PyTorch: {torch.__version__}')

def build_ham(N):
    h = Hamiltonian(num_qubits=N)
    for i in range(N):
        j = (i+1) % N
        p = ['I']*N; p[i]='Z'; p[j]='Z'
        h.add_pauli(''.join(p), 0.5)
    h.add_pauli('I'*N, 1.0)
    return h

def fd_grad(theta, circuit, h):
    g = torch.zeros(theta.numel(), device=device)
    for k in range(theta.numel()):
        tp = theta.clone(); tp[k] += 1e-4
        fp = ev_exact(circuit.build_tensor(tp), h, device=device).item()
        tp[k] -= 2e-4
        fm = ev_exact(circuit.build_tensor(tp), h, device=device).item()
        g[k] = (fp - fm) / 2e-4
    return g

def bench(fn, warmup=3, repeats=5):
    for _ in range(warmup): fn()
    if device == 'cuda': torch.cuda.synchronize()
    times = []
    for _ in range(repeats):
        if device == 'cuda': torch.cuda.synchronize()
        t0 = time.perf_counter()
        fn()
        if device == 'cuda': torch.cuda.synchronize()
        times.append((time.perf_counter() - t0) * 1000)
    return np.median(times)

## 1. Accuracy: Ring vs Chain Topology
Test autograd on both **ring** (periodic, CNOT wraps around) and **chain** (open boundary) topologies.

In [ ]:
torch.manual_seed(42)
print(f'{"topology":<45} {"cos(ad,fd)":>10} {"cos(ps,fd)":>10}')
print('=' * 70)

for N, chi in [(8, 4), (8, 10), (12, 4), (12, 10), (16, 4)]:
    for topo_name, make_cnots in [
        ('ring  (CNOT i->(i+1)%N)', lambda c, N: [c.cx(i, (i+1)%N) for i in range(N)]),
        ('chain (CNOT i->i+1)',      lambda c, N: [c.cx(i, i+1) for i in range(N-1)]),
    ]:
        c = Circuit(num_qubit=N, rank=chi, device=device)
        for i in range(N): c.h(i)
        for _ in range(2):
            make_cnots(c, N)
            for i in range(N): c.ry(i); c.rz(i)
        h = build_ham(N)
        theta = torch.randn(c.params_size, device=device)

        gf = fd_grad(theta, c, h)
        ga = autograd_gradient(theta, c, h)
        gp = batch_gradient(theta, c, h, 8, 0, 0.5*3.14159, 1, 0, False,
                           MeasureMethod.EFFICIENT_CONTRACTION)

        ca = torch.nn.functional.cosine_similarity(ga.unsqueeze(0), gf.unsqueeze(0)).item()
        cp = torch.nn.functional.cosine_similarity(gp.unsqueeze(0), gf.unsqueeze(0)).item()
        ok = 'PASS' if ca > 0.95 else 'FAIL'
        print(f'N={N:>2} chi={chi:>2} {topo_name:<28} {ca:>10.4f} {cp:>10.4f} {ok}')
    print()

## 2. Accuracy: Various Circuit Types
HEA, QAOA, and deep circuits with ring topology.

In [ ]:
torch.manual_seed(42)
print(f'{"circuit":<40} {"cos(ad,fd)":>10} {"cos(ps,fd)":>10}')
print('=' * 65)

# HEA
print('--- HEA (H + [ring CNOT + RY,RZ] x L) ---')
for N, chi, L in [(8,4,2), (8,10,2), (12,10,2), (16,4,2)]:
    c = Circuit(num_qubit=N, rank=chi, device=device)
    for i in range(N): c.h(i)
    for _ in range(L):
        for i in range(N): c.cx(i, (i+1)%N)
        for i in range(N): c.ry(i); c.rz(i)
    h = build_ham(N)
    theta = torch.randn(c.params_size, device=device)
    gf = fd_grad(theta, c, h)
    ga = autograd_gradient(theta, c, h)
    gp = batch_gradient(theta, c, h, 8, 0, 0.5*3.14159, 1, 0, False, MeasureMethod.EFFICIENT_CONTRACTION)
    ca = torch.nn.functional.cosine_similarity(ga.unsqueeze(0), gf.unsqueeze(0)).item()
    cp = torch.nn.functional.cosine_similarity(gp.unsqueeze(0), gf.unsqueeze(0)).item()
    print(f'HEA N={N:>2} chi={chi:>2} L={L} P={c.params_size:>3}              {ca:>10.4f} {cp:>10.4f} {"PASS" if ca>0.95 else "FAIL"}')

# QAOA
print('\n--- QAOA (H + [CNOT-RZ-CNOT per edge + RX] x p) ---')
for N, chi, p in [(8,4,2), (8,10,2), (12,10,2)]:
    c = Circuit(num_qubit=N, rank=chi, device=device)
    for i in range(N): c.h(i)
    for _ in range(p):
        for i in range(N):
            j = (i+1) % N
            c.cx(i, j); c.rz(j); c.cx(i, j)
        for i in range(N): c.rx(i)
    h = build_ham(N)
    theta = torch.randn(c.params_size, device=device)
    gf = fd_grad(theta, c, h)
    ga = autograd_gradient(theta, c, h)
    gp = batch_gradient(theta, c, h, 8, 0, 0.5*3.14159, 1, 0, False, MeasureMethod.EFFICIENT_CONTRACTION)
    ca = torch.nn.functional.cosine_similarity(ga.unsqueeze(0), gf.unsqueeze(0)).item()
    cp = torch.nn.functional.cosine_similarity(gp.unsqueeze(0), gf.unsqueeze(0)).item()
    print(f'QAOA N={N:>2} chi={chi:>2} p={p} P={c.params_size:>3}              {ca:>10.4f} {cp:>10.4f} {"PASS" if ca>0.95 else "FAIL"}')

## 3. Speed Benchmark

In [ ]:
torch.manual_seed(42)
print(f'{"circuit":<30} {"autograd":>10} {"param_shift":>12} {"speedup":>8}')
print('-' * 65)

for N, chi, L in [(4,4,2), (8,4,2), (8,10,2), (12,4,2), (12,10,2), (16,4,2), (16,10,2), (20,4,2), (24,4,2)]:
    c = Circuit(num_qubit=N, rank=chi, device=device)
    for i in range(N): c.h(i)
    for _ in range(L):
        for i in range(N): c.cx(i, (i+1)%N)
        for i in range(N): c.ry(i); c.rz(i)
    h = build_ham(N)
    theta = torch.randn(c.params_size, device=device)

    ms_ad = bench(lambda: autograd_gradient(theta, c, h))
    ms_ps = bench(lambda: batch_gradient(theta, c, h, 8, 0, 0.5*3.14159, 1, 0, False, MeasureMethod.EFFICIENT_CONTRACTION))
    sp = ms_ps / ms_ad
    print(f'HEA N={N:>2} chi={chi:>2} L={L} P={c.params_size:>3} {ms_ad:>8.0f}ms {ms_ps:>10.0f}ms {sp:>7.2f}x')

## 4. Large Circuit (where autograd wins on big GPU)
When P >> batch_size, parameter-shift needs many chunks but autograd still does 1 pass.

In [ ]:
torch.manual_seed(42)
print(f'{"circuit":<35} {"ad (ms/it)":>12} {"ps (ms/it)":>12} {"speedup":>8}')
print('-' * 72)

for N, chi, L in [(20, 10, 2), (24, 10, 2), (32, 4, 2), (16, 10, 3)]:
    c = Circuit(num_qubit=N, rank=chi, device=device)
    for i in range(N): c.h(i)
    for _ in range(L):
        for i in range(N): c.cx(i, (i+1)%N)
        for i in range(N): c.ry(i); c.rz(i)
    h = build_ham(N)
    theta = torch.randn(c.params_size, device=device)
    opt = Optimizer(torch.optim.Adam, {'lr': 0.05})
    iters = 5

    grad_ad = AutogradGradient(); grad_ad._verbose = False
    t0 = time.time()
    minimize(c, theta.clone(), h, opt, grad_ad, iteration=iters, best_value_method='contraction')
    t_ad = time.time() - t0

    grad_ps = BatchParameterShiftGradient(shift=0.5*3.14159, batch_size=None, shots=0,
                                           measure_method=MeasureMethod.EFFICIENT_CONTRACTION, depth=1)
    grad_ps._verbose = False
    t0 = time.time()
    minimize(c, theta.clone(), h, opt, grad_ps, iteration=iters, best_value_method='contraction')
    t_ps = time.time() - t0

    ad_ms = t_ad/iters*1000; ps_ms = t_ps/iters*1000
    sp = ps_ms / ad_ms
    print(f'HEA N={N:>2} chi={chi:>2} L={L} P={c.params_size:>4}     {ad_ms:>10.0f}ms {ps_ms:>10.0f}ms {sp:>7.2f}x')

## 5. Optimization Convergence

In [ ]:
torch.manual_seed(42)
N, chi, L = 12, 10, 2
c = Circuit(num_qubit=N, rank=chi, device=device)
for i in range(N): c.h(i)
for _ in range(L):
    for i in range(N): c.cx(i, (i+1)%N)
    for i in range(N): c.ry(i); c.rz(i)
h = build_ham(N)
theta = torch.randn(c.params_size, device=device)
opt = Optimizer(torch.optim.Adam, {'lr': 0.05})
iters = 50

print(f'=== AutogradGradient (N={N}, chi={chi}, L={L}, P={c.params_size}) ===')
grad_ad = AutogradGradient()
t0 = time.time()
r_ad = minimize(c, theta.clone(), h, opt, grad_ad, iteration=iters, best_value_method='contraction')
t_ad = time.time() - t0

print(f'\n=== BatchParameterShiftGradient ===')
grad_ps = BatchParameterShiftGradient(shift=0.5*3.14159, batch_size=None, shots=0,
                                       measure_method=MeasureMethod.EFFICIENT_CONTRACTION, depth=1)
t0 = time.time()
r_ps = minimize(c, theta.clone(), h, opt, grad_ps, iteration=iters, best_value_method='contraction')
t_ps = time.time() - t0

print(f'\nAutograd:    {t_ad:.1f}s ({t_ad/iters*1000:.0f}ms/iter)')
print(f'ParamShift:  {t_ps:.1f}s ({t_ps/iters*1000:.0f}ms/iter)')
print(f'Speedup:     {t_ps/t_ad:.2f}x')

In [ ]:
import matplotlib.pyplot as plt

ev_ad = [v.item() if hasattr(v, 'item') else float(v) for v in r_ad[1]]
ev_ps = [v.item() if hasattr(v, 'item') else float(v) for v in r_ps[1]]

fig, ax = plt.subplots(1, 1, figsize=(8, 5))
ax.plot(ev_ad, label=f'Autograd ({t_ad:.1f}s)', linewidth=2)
ax.plot(ev_ps, label=f'Param-shift ({t_ps:.1f}s)', linewidth=2, linestyle='--')
ax.set_xlabel('Iteration')
ax.set_ylabel('Energy')
ax.set_title(f'VQE Convergence (ring topology): N={N}, chi={chi}, L={L}')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()